In [46]:
from datasets import load_dataset
def load_wildguard():
    wildguard_test = load_dataset("allenai/wildguardmix", "wildguardtest")["test"]
    wildguard_test = wildguard_test.to_pandas()
    wildguard_test = wildguard_test.dropna()

    prompts = wildguard_test["prompt"].values.tolist()
    labels = wildguard_test["prompt_harm_label"].values.tolist()
    map2id = {"unharmful": 0, "harmful": 1}
    labels= [map2id[x] for x in labels]
    return prompts,labels,wildguard_test

In [47]:
real_prompt, real_labels,df = load_wildguard()

Using the latest cached version of the dataset since allenai/wildguardmix couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wildguardtest' at /home/unica/.cache/huggingface/datasets/allenai___wildguardmix/wildguardtest/0.0.0/d29c47f41c8b51348b5c8e8c81c039b3132b66d1 (last modified on Mon May 26 10:32:15 2025).


In [48]:
from glob import glob
import pandas as pd
from sklearn.metrics import accuracy_score,f1_score

def wildguard_scores(pred_df):
    wild_adv=df[df["adversarial"]==True]    
    pred_adv=pred_df[pred_df["text"].isin(wild_adv["prompt"])]    
    wild_van=df[df["adversarial"]==False]
    pred_van=pred_df[pred_df["text"].isin(wild_van["prompt"])]
    
    overall=accuracy_f1(pred_df["real"].values.tolist(),pred_df["pred"].values.tolist())
    vanilla=accuracy_f1(pred_van["real"].values.tolist(),pred_van["pred"].values.tolist())
    adversarial=accuracy_f1(pred_adv["real"].values.tolist(),pred_adv["pred"].values.tolist())
    return {"Overall":overall,"Vanilla":vanilla,"Adversarial":adversarial}
def accuracy_f1(real,preds):
    return {"ACC":round(accuracy_score(real, preds),2),"F1":round(f1_score(real, preds),2)}

def compute_scores(folder):    
    path=f"../output/parsed/{folder}/"
    models=os.listdir(path)
    res={k:[] for k in models}
    for model in models:  
        preds=glob(f"../output/parsed/{folder}/{model}/*.json")
        
        for pred in preds:
            dataset=pred.split("/")[-1].replace(".json","")
           
                
            pred_df=pd.read_json(pred)
            if dataset!="WildGuard":
                preds=pred_df["pred"].values.tolist()
                real=pred_df["real"].values.tolist()
                
                res[model].append({dataset:accuracy_f1(real,preds)})
            else:
                res[model].append({dataset:wildguard_scores(pred_df)})
    
    
    for model,results in res.items():
        print("MODEL",model)
        for item in results:
            print(item)
        print("*"*40)

### ZERO-SHOT


In [52]:
compute_scores("ZERO")

MODEL llama3.3-70
{'Remedy': {'ACC': 0.92, 'F1': 0.92}}
{'Aegis': {'ACC': 0.82, 'F1': 0.84}}
{'WildGuard': {'Overall': {'ACC': 0.89, 'F1': 0.86}, 'Vanilla': {'ACC': 0.92, 'F1': 0.91}, 'Adversarial': {'ACC': 0.84, 'F1': 0.8}}}
{'OrBench': {'ACC': 0.5, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.96, 'F1': 0.7}}
****************************************
MODEL .ipynb_checkpoints
****************************************
MODEL llama3.1-8
{'Remedy': {'ACC': 0.88, 'F1': 0.87}}
{'Aegis': {'ACC': 0.75, 'F1': 0.77}}
{'WildGuard': {'Overall': {'ACC': 0.79, 'F1': 0.71}, 'Vanilla': {'ACC': 0.87, 'F1': 0.84}, 'Adversarial': {'ACC': 0.71, 'F1': 0.5}}}
{'OrBench': {'ACC': 0.77, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.95, 'F1': 0.56}}
****************************************
MODEL gpt4-mini
{'Remedy': {'ACC': 0.9, 'F1': 0.9}}
{'Aegis': {'ACC': 0.83, 'F1': 0.86}}
{'WildGuard': {'Overall': {'ACC': 0.88, 'F1': 0.86}, 'Vanilla': {'ACC': 0.91, 'F1': 0.9}, 'Adversarial': {'ACC': 0.84, 'F1': 0.82}}}
{'OrBench': {'ACC': 0.1

### FINE-TUNED

In [50]:
compute_scores("FT")

MODEL llama3.1-8
{'Remedy': {'ACC': 0.96, 'F1': 0.96}}
{'Aegis': {'ACC': 0.85, 'F1': 0.88}}
{'WildGuard': {'Overall': {'ACC': 0.87, 'F1': 0.85}, 'Vanilla': {'ACC': 0.94, 'F1': 0.93}, 'Adversarial': {'ACC': 0.8, 'F1': 0.75}}}
{'OrBench': {'ACC': 0.69, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.94, 'F1': 0.68}}
****************************************
MODEL llama3.2-3
{'Remedy': {'ACC': 0.96, 'F1': 0.96}}
{'Aegis': {'ACC': 0.79, 'F1': 0.82}}
{'WildGuard': {'Overall': {'ACC': 0.84, 'F1': 0.81}, 'Vanilla': {'ACC': 0.91, 'F1': 0.89}, 'Adversarial': {'ACC': 0.76, 'F1': 0.71}}}
{'OrBench': {'ACC': 0.83, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.95, 'F1': 0.72}}
****************************************


### GUARDIANS

In [51]:
compute_scores("GUARD")

MODEL shieldgemma
{'Remedy': {'ACC': 0.82, 'F1': 0.79}}
{'Aegis': {'ACC': 0.74, 'F1': 0.76}}
{'WildGuard': {'Overall': {'ACC': 0.72, 'F1': 0.56}, 'Vanilla': {'ACC': 0.76, 'F1': 0.66}, 'Adversarial': {'ACC': 0.68, 'F1': 0.41}}}
{'OrBench': {'ACC': 0.74, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.96, 'F1': 0.67}}
****************************************
MODEL wildguard
{'Remedy': {'ACC': 0.96, 'F1': 0.96}}
{'Aegis': {'ACC': 0.87, 'F1': 0.89}}
{'WildGuard': {'Overall': {'ACC': 0.9, 'F1': 0.89}, 'Vanilla': {'ACC': 0.93, 'F1': 0.92}, 'Adversarial': {'ACC': 0.88, 'F1': 0.85}}}
{'OrBench': {'ACC': 0.26, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.93, 'F1': 0.65}}
****************************************
MODEL llamaguard
{'Remedy': {'ACC': 0.86, 'F1': 0.85}}
{'Aegis': {'ACC': 0.71, 'F1': 0.72}}
{'WildGuard': {'Overall': {'ACC': 0.82, 'F1': 0.77}, 'Vanilla': {'ACC': 0.89, 'F1': 0.87}, 'Adversarial': {'ACC': 0.75, 'F1': 0.62}}}
{'OrBench': {'ACC': 0.81, 'F1': 0.0}}
{'ToxicChat': {'ACC': 0.92, 'F1': 0.48}}
****